# 🔆 Teja — Stage 5: Real Data + BPE + Scale
**Built from zero. Trained to shine.**  
Created by Sabith, Nilambur, Kerala.

---

### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier is fine)
2. Run cells top to bottom

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {mem:.1f} GB')
else:
    print('No GPU detected — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: (Optional) Mount Google Drive to persist checkpoints ─────────
# If you skip this, checkpoints live in /content/ and are lost when
# the Colab session ends. Mount Drive to keep them permanently.
#
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Clone the Teja repo ──────────────────────────────────────────
import os

REPO_URL = 'https://github.com/sabithh/teja'  # ← your GitHub repo URL
REPO_DIR = '/content/teja'

if os.path.exists(REPO_DIR):
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls

In [ ]:
# ── Cell 4: Install Stage 5 dependencies ─────────────────────────────────
!pip install -q tiktoken datasets tqdm
print('Done.')

In [ ]:
# ── Cell 5: (Optional) Symlink checkpoints to Drive ──────────────────────
# If you mounted Drive in Cell 2, run this so checkpoints survive
# after the session ends.
#
# import os
# drive_ckpt = '/content/drive/MyDrive/teja_checkpoints'
# os.makedirs(drive_ckpt, exist_ok=True)
# if not os.path.islink('/content/teja/checkpoints'):
#     os.symlink(drive_ckpt, '/content/teja/checkpoints')
# print('Checkpoints → Drive:', drive_ckpt)

In [ ]:
# ── Cell 6: Run Stage 5! ──────────────────────────────────────────────────
# This will:
#   1. Stream + tokenize OpenWebText (runs once, ~15 min)
#   2. Train for 50,000 steps (~3-5 hours on T4)
#   3. Save best checkpoint automatically
#   4. Print generated text samples
#
# TIP: If Colab disconnects, re-run — data prep is skipped if already done.
#      Training resumes from scratch unless you add checkpoint loading.

!python stages/stage5_scale.py

In [ ]:
# ── Cell 7: Quick inference (generate text from saved checkpoint) ─────────
import sys, os
sys.path.insert(0, '/content/teja')

import torch
import tiktoken
from stages.stage5_scale import TejaGPT, block_size, n_embd, n_head, n_layer, dropout, vocab_size

device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc    = tiktoken.get_encoding('gpt2')

ckpt_path = '/content/teja/checkpoints/teja_stage5_best.pt'
ckpt = torch.load(ckpt_path, map_location=device)

model = TejaGPT().to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded checkpoint — val loss: {ckpt['val_loss']:.4f}")

# Change this prompt and re-run the cell
PROMPT      = "The future of artificial intelligence"
MAX_TOKENS  = 200
TEMPERATURE = 0.8
TOP_K       = 200

ids      = enc.encode(PROMPT)
context  = torch.tensor([ids], dtype=torch.long, device=device)
with torch.no_grad():
    out = model.generate(context, MAX_TOKENS, temperature=TEMPERATURE, top_k=TOP_K)
print(enc.decode(out[0].tolist()))